In [ ]:
# MarketMind Intelligence Platform V1
# Author: Sharique Mohammad
# Date: April 23, 2026

# MarketMind V1 - Cross-Dataset Analysis

**Notebook:** 06_cross_dataset_analysis.ipynb  
**Purpose:** Correlate market data with macro indicators and SEC filings

---

## Overview

This notebook performs cross-dataset analysis:
- Market performance vs economic indicators
- Stock price reactions to SEC filings
- Correlation between unemployment and market trends
- Interest rate impact on stock prices
- Multi-dataset timeline visualization

**Datasets:**
- gold.ohlcv_bars (market data)
- gold.macro_indicators (economic data)
- gold.sec_filings (regulatory filings)

In [ ]:
# Import libraries
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from scipy import stats

# Set style
plt.style.use('seaborn-v0_8-whitegrid')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print('Libraries imported successfully!')

## 1. Load All Datasets

In [ ]:
# PostgreSQL connection
DB_CONFIG = {
    'host': '172.31.32.1',
    'port': 5432,
    'database': 'marketmind_v1',
    'user': 'postgres',
    'password': '0940'
}

conn = psycopg2.connect(**DB_CONFIG)

# Load market data
df_market = pd.read_sql("""
    SELECT ticker, date, close, volume
    FROM gold.ohlcv_bars
    ORDER BY date, ticker
""", conn)
df_market['date'] = pd.to_datetime(df_market['date'])

# Load macro data
df_macro = pd.read_sql("""
    SELECT indicator_name, date, value
    FROM gold.macro_indicators
    WHERE value IS NOT NULL
    ORDER BY date, indicator_name
""", conn)
df_macro['date'] = pd.to_datetime(df_macro['date'])

# Load SEC filings
df_filings = pd.read_sql("""
    SELECT ticker, filing_date, form_type
    FROM gold.sec_filings
    WHERE ticker != 'UNKNOWN'
    ORDER BY filing_date
""", conn)
df_filings['filing_date'] = pd.to_datetime(df_filings['filing_date'])

conn.close()

print(f'Market Data: {len(df_market):,} records')
print(f'Macro Data: {len(df_macro):,} records')
print(f'SEC Filings: {len(df_filings):,} records')
print('\nAll datasets loaded successfully')

## 2. Market Index vs Unemployment Rate

In [ ]:
# Create market index (average of all stocks)
market_index = df_market.groupby('date')['close'].mean().reset_index()
market_index.columns = ['date', 'market_index']

# Get unemployment rate
unemployment = df_macro[df_macro['indicator_name'] == 'US_UNEMPLOYMENT_RATE'][['date', 'value']]
unemployment.columns = ['date', 'unemployment_rate']

# Merge datasets on date (using merge_asof for nearest date matching)
combined = pd.merge_asof(
    market_index.sort_values('date'),
    unemployment.sort_values('date'),
    on='date',
    direction='backward'
)
combined = combined.dropna()

if len(combined) > 0:
    # Plot dual-axis chart
    fig, ax1 = plt.subplots(figsize=(14, 7))
    
    # Market index on left axis
    color = 'steelblue'
    ax1.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Market Index (Avg Stock Price)', color=color, fontsize=12, fontweight='bold')
    ax1.plot(combined['date'], combined['market_index'], color=color, linewidth=2.5, label='Market Index')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, alpha=0.3)
    
    # Unemployment on right axis
    ax2 = ax1.twinx()
    color = 'darkred'
    ax2.set_ylabel('Unemployment Rate (%)', color=color, fontsize=12, fontweight='bold')
    ax2.plot(combined['date'], combined['unemployment_rate'], color=color, linewidth=2.5, 
             linestyle='--', label='Unemployment Rate')
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title('Market Performance vs Unemployment Rate', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Calculate correlation
    if len(combined) >= 2:
        corr, pval = stats.pearsonr(combined['market_index'], combined['unemployment_rate'])
        print(f'Correlation: {corr:.3f}')
        print(f'P-value: {pval:.4f}')
        if abs(corr) > 0.5:
            direction = 'negative' if corr < 0 else 'positive'
            print(f'Result: Strong {direction} correlation detected')
        else:
            print('Result: Weak correlation')
else:
    print('Insufficient overlapping data for analysis')

## 3. Market Index vs Federal Funds Rate

In [ ]:
# Get federal funds rate
fed_rate = df_macro[df_macro['indicator_name'] == 'US_FEDERAL_FUNDS_RATE'][['date', 'value']]
fed_rate.columns = ['date', 'fed_rate']

# Merge with market index
combined_fed = pd.merge_asof(
    market_index.sort_values('date'),
    fed_rate.sort_values('date'),
    on='date',
    direction='backward'
)
combined_fed = combined_fed.dropna()

if len(combined_fed) > 0:
    # Plot dual-axis chart
    fig, ax1 = plt.subplots(figsize=(14, 7))
    
    # Market index on left axis
    color = 'steelblue'
    ax1.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Market Index (Avg Stock Price)', color=color, fontsize=12, fontweight='bold')
    ax1.plot(combined_fed['date'], combined_fed['market_index'], color=color, linewidth=2.5)
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.grid(True, alpha=0.3)
    
    # Fed rate on right axis
    ax2 = ax1.twinx()
    color = 'darkgreen'
    ax2.set_ylabel('Federal Funds Rate (%)', color=color, fontsize=12, fontweight='bold')
    ax2.plot(combined_fed['date'], combined_fed['fed_rate'], color=color, linewidth=2.5, linestyle='--')
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title('Market Performance vs Interest Rates', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Calculate correlation
    if len(combined_fed) >= 2:
        corr, pval = stats.pearsonr(combined_fed['market_index'], combined_fed['fed_rate'])
        print(f'Correlation: {corr:.3f}')
        print(f'P-value: {pval:.4f}')
        if abs(corr) > 0.5:
            direction = 'negative' if corr < 0 else 'positive'
            print(f'Result: Strong {direction} correlation detected')
        else:
            print('Result: Weak correlation')
else:
    print('Insufficient overlapping data for analysis')

## 4. Stock Price Reaction to SEC Filings

In [ ]:
# Analyze price changes around filing dates
if len(df_filings) > 0:
    # Select a ticker with filings (e.g., first available)
    sample_ticker = df_filings['ticker'].value_counts().index[0]
    
    # Get filings for sample ticker
    ticker_filings = df_filings[df_filings['ticker'] == sample_ticker].copy()
    
    # Get price data for sample ticker
    ticker_prices = df_market[df_market['ticker'] == sample_ticker].copy()
    
    # Plot price chart with filing markers
    fig, ax = plt.subplots(figsize=(14, 7))
    
    ax.plot(ticker_prices['date'], ticker_prices['close'], linewidth=2, color='steelblue', label='Stock Price')
    
    # Add filing markers
    for _, filing in ticker_filings.iterrows():
        filing_date = filing['filing_date']
        form_type = filing['form_type']
        
        # Find nearest price
        nearest = ticker_prices[ticker_prices['date'] >= filing_date].head(1)
        if len(nearest) > 0:
            price = nearest.iloc[0]['close']
            color = 'red' if form_type == '10-K' else 'orange' if form_type == '10-Q' else 'green'
            ax.axvline(filing_date, color=color, linestyle='--', alpha=0.5, linewidth=1.5)
            ax.scatter(filing_date, price, color=color, s=100, zorder=5, 
                      label=form_type if form_type not in ax.get_legend_handles_labels()[1] else '')
    
    ax.set_xlabel('Date', fontsize=12, fontweight='bold')
    ax.set_ylabel('Stock Price ($)', fontsize=12, fontweight='bold')
    ax.set_title(f'{sample_ticker} Stock Price with SEC Filing Events', fontsize=14, fontweight='bold')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f'Analysis for: {sample_ticker}')
    print(f'Total filings in period: {len(ticker_filings)}')
    print('Filing markers:')
    print('  Red dashed line: 10-K (Annual Report)')
    print('  Orange dashed line: 10-Q (Quarterly Report)')
    print('  Green dashed line: 8-K (Current Report)')
else:
    print('No filings data available for analysis')

## 5. Multi-Indicator Correlation Matrix

In [ ]:
# Create wide-format dataset with all indicators
macro_pivot = df_macro.pivot(index='date', columns='indicator_name', values='value')

# Add market index
combined_all = pd.merge(
    market_index.set_index('date'),
    macro_pivot,
    left_index=True,
    right_index=True,
    how='inner'
)

# Drop rows with NaN
combined_all = combined_all.dropna()

if len(combined_all) > 0:
    # Calculate correlation matrix
    corr_matrix = combined_all.corr()
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                square=True, linewidths=1, cbar_kws={'label': 'Correlation'},
                ax=ax)
    
    ax.set_title('Cross-Dataset Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()
    
    # Print strongest correlations with market index
    market_corr = corr_matrix['market_index'].sort_values(ascending=False)
    print('Correlations with Market Index:')
    print('=' * 60)
    for indicator, corr in market_corr.items():
        if indicator != 'market_index':
            print(f'{indicator:.<45} {corr:>6.3f}')
    print('=' * 60)
else:
    print('Insufficient overlapping data for correlation analysis')

## 6. Timeline Integration - All Events

In [ ]:
# Create integrated timeline for Q1 2026
q1_start = pd.Timestamp('2026-01-01')
q1_end = pd.Timestamp('2026-04-30')

# Filter data for Q1 2026
market_q1 = df_market[(df_market['date'] >= q1_start) & (df_market['date'] <= q1_end)]
macro_q1 = df_macro[(df_macro['date'] >= q1_start) & (df_macro['date'] <= q1_end)]
filings_q1 = df_filings[(df_filings['filing_date'] >= q1_start) & (df_filings['filing_date'] <= q1_end)]

# Create market index for Q1
market_idx_q1 = market_q1.groupby('date')['close'].mean().reset_index()

# Plot integrated timeline
fig, ax = plt.subplots(figsize=(16, 8))

# Plot market index
ax.plot(market_idx_q1['date'], market_idx_q1['close'], linewidth=2.5, color='steelblue', label='Market Index')

# Add macro indicator releases (vertical lines)
for indicator in macro_q1['indicator_name'].unique():
    ind_data = macro_q1[macro_q1['indicator_name'] == indicator]
    for _, row in ind_data.iterrows():
        ax.axvline(row['date'], color='gray', linestyle=':', alpha=0.3, linewidth=1)

# Add SEC filing markers
for _, filing in filings_q1.iterrows():
    color = 'red' if filing['form_type'] == '10-K' else 'orange' if filing['form_type'] == '10-Q' else 'green'
    ax.axvline(filing['filing_date'], color=color, linestyle='--', alpha=0.6, linewidth=2)

ax.set_xlabel('Date (Q1 2026)', fontsize=12, fontweight='bold')
ax.set_ylabel('Market Index', fontsize=12, fontweight='bold')
ax.set_title('Q1 2026 Integrated Timeline - Market, Macro, and SEC Events', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(['Market Index', '10-K Filings', '10-Q Filings', '8-K Filings', 'Macro Releases'], loc='best')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('Timeline Legend:')
print('  Blue line: Market Index (average stock price)')
print('  Gray dotted lines: Macro indicator releases')
print('  Red dashed lines: 10-K filings')
print('  Orange dashed lines: 10-Q filings')
print('  Green dashed lines: 8-K filings')